# Extending Tomobase

The purpose of this tutorial is to show users how to extend the TomoBase library with new functions.

## Custom Register Basics

The Register class stores objects that can be utilizes throughout code built on top of Tomobase. It functions similarly to a strictly typed dictionary with extra functions. In the example below we have created a register called blob. The blob register has a named key and takes in an ImageAbstract (A base class, used for building image data such as sinograms, and volumes). There are a number of registers used in TomoBase, see the [API](index.md) for details. 


In [1]:
from tomobase.core import logger
import logging

logger.setLevel(logging.INFO)

import tomobase
from tomobase.core.base_classes import Registry, ImageAbstract
from tomobase import procedures
# if new registries are needed, they can be specified with a name and a type.
blob = Registry(str, ImageAbstract)


From there, a new item can be added by calling the registor method.

In [2]:

from tomobase.core import base_classes


#name is a display name for Napari it defaults to the name of the object
@blob.register(name="Test Image")
class TestImage(base_classes.ImageAbstract):
    def __init__(self, name, data, metadata=None):
        super().__init__(name, data, pixel_size=1, metadata=metadata)
        
# images.register(TestImage) also works if you dont want to decorate.

The object can then be used by calling it from the register. The example blob is an empty copy of the prominantly used images register.

In [3]:
import numpy as np
import xarray as xr

data = np.zeros((10, 10, 10))
data = xr.DataArray(data, dims=["x", "y", "z"], coords={"z": np.arange(10), "y": np.arange(10), "x": np.arange(10)})
new_object = blob.TestImage("Test Stuff", data)

Custom items and registries are not persistently stored. If you close your instance, the item will need to be reregistered again. You can just rerun the cell. However, for more complex customizations it may be worthwile to make libraries or scripts. The following functions add libraries or scripts for tomobase to search on startup. Do not run these examples without a real file or package otherwise tomobase will attempt to add the nonexistent package to its config. This will probably not cause errors but for developers, commits will accumulate unnecssary config settings. 

In [ ]:

tomobase.packages.add_package('nameofpackage') # add to __init__.py of the package.

In [ ]:
tomobase.packages.add_path('nameofpackage.py') #only supports .py files and is currently experimental.

## Initialized Registers

Sometimes, you may want to modify an item in a register. This is supported using an initialization function.



In [ ]:

def decorate_class(cls):
    if 'state_changed' in cls._tomobase_kwargs:
        cls.new_state = False
    else:
        cls.new_state = True
    return cls

blob.set_initialization_function(decorate_class)

try:
    print(blob.TestImage.new_state) 
except AttributeError:
    print("Attribute not found")

The new_state attribute is not found in our TestImage. This is because the register has not been initialized. An example of the initialization can be seen here. For the registers in TomoBase, this is done by default when tomobase.bootstrap() is called. If you have a library added with add_package(), perform the initialization in bootstrap() and TomoBase will find the function in your library when tomobase.bootstrap() is called.

In [ ]:

blob.initialize()
print(blob.TestImage.new_state) 

@blob.register(name="Test Image2", state_changed=True)
class TestImage2(base_classes.ImageAbstract):
    def __init__(self, name, data, metadata=None):
        super().__init__(name, data, pixel_size=1, metadata=metadata)
        
print(blob.TestImage2.new_state) 
print("state_change allows you to modify the state on registration.")

## Hierarchical Registers

A hierarchical register can be used to categorize registered items when their are too many items. For an item in a HierarchicalRegister is stored with a name and value (integer 0 and 255). The following code provides an example of how to create a hierarchy with the following structure.

- Test Function      (0x01 00 00 00 00)
  - Middle Tier      (0x01 01 00 00 00)
    - Bottom Tier 1  (0x01 01 02 00 00)
    - Bottom Tier 2  (0x01 01 03 00 00)


In [ ]:
from tomobase.core.base_classes import HierarchicalRegistry

hierarchy = HierarchicalRegistry(str, int)
hierarchy.add_hierarchy(name="Test Functions", value=1)
hierarchy.add_hierarchy(name="Middle Tier", value=1, parent="Test Functions")
hierarchy.add_hierarchy(name="Bottom Tier 1", value=2, parent="Middle Tier")
hierarchy.add_hierarchy(name="Bottom Tier 2", value=3, parent="Middle Tier")


In [ ]:
from typing import  Callable

blob2 = Registry(str, Callable)
blob2.set_hierarchy(hierarchy, hierarchy["Test Functions"])

@blob2.register(name="Test Image", category=hierarchy["Bottom Tier 1"])
def test_func():
    print("This is a test function.")
       
print(blob2.middle_tier.bottom_tier_1.test_func)
print(blob2.middle_tier.bottom_tier_1.test_func())

## Procedures

Procedures is the main registry for storing functions that operate on image data. This registry is discussed further in the [APIs](..\apis\index.md). Essentially, procedures uses the features outlined above to modify a function for the following tasks
- Providing experimental history of samples.
- Managing the GPU/CPU
- Managing wether objects are copied into meemory or worked inplace.
- Saving structured results for analysis
- Managing wether to provide additional return data for quantifying intermediate steps 

In [ ]:
from tomobase import categories
from tomobase import core

subcategory = categories.add_hierarchy("Test Functions 2", value=32, parent="Image Processing") 
# returns the category code of the newly created subcategory in Image Processing
print(hierarchy.get_key(subcategory)) # returns the name of the subcategory

#categories["Test Functions"] is equivalent to the subcategory variable
@procedures.register(name = "Random Multiplier", category=subcategory, use_numpy=True) 
def blobby_function(image: core.base_classes.ImageAbstract): 
    #Type hints are optional but good practice. 
    #ImageAbstract is the base class for all images in tomobase, so any image type can be used as input.
    #use numpy overrides the default GPU behavior to ignore CUPY if specified.
    """Test Function for adding to the library. 
    
    Documentation strings like this one are optional but 
    
    """
    xp = core.get_xp(image.data) #gets the module used for the data 
    core.logger.info("This is a test function.")
    
    rand_num = xp.random.rand() #generates a random number using the same module as the data.
    #core.data_classes.Measurement(name="Multipliers", vau)
    
    image.data = image.data.astype(xp.float32)
    image.data *= rand_num
    return image, rand_num

The function can then be called from the register, as shown below. Once bootstrap is called all future functions added will be initialized. Hence, registered items can be added before or after bootstrap calls.

In [ ]:
import tomobase

tomobase.bootstrap()
phantom = procedures.phantoms.get_nanocage()
phantom = procedures.image_processing.test_functions_2.blobby_function(phantom)

## Documentation

All items registered by default in TomoBase are shown in the [APIs](..\apis\index.md). However, custom functions cannot be added to the documentation. If you ever have difficulty finding a function, all registers have a help function. Call help or log the registry and this will print all the contents of the registry. Whilst documenting items is optional, so long as functions and classes are documented with a docstring, they will appear fully documented in the registry. TomoBase uses google docstring conventions.

In [ ]:
# This needs to be fixed
procedures.help()
# alternatively logger.info(procedures)

Its worth noting, if you import a function inside a script like this one, it is not persistent. The module will not be found by TomoBase when TomoBase is restarted. Of course, you can always restart the script. However, you can also register the module with tomobase. This will allow tomobase to find the module on start up. The following command allows users to register a module or script. These registration functions will 